# Vincent Straussberger

## RAG Model

This notebook uses retrieval-augmented generation with Qwen and RIS law sources to answer Austrian tax questions with supporting context.

Key facts:
- model: `Qwen/Qwen3-0.6B`
- retrieval sources: RIS seed URLs and extracted law text
- benchmark input: `dataset_clean.csv`
- output: one submission CSV with `id,answer`


## How the code works

- loads RIS source links and builds a local retrieval corpus
- finds relevant passages for each benchmark question
- gives the retrieved context to the model before generation
- saves the final CSV in this folder


In [ ]:
print('Skip this cell if your packages are already installed.')
print('Minimal install for this notebook: python -m pip install pandas numpy torch transformers sentence-transformers beautifulsoup4 pypdf requests tqdm')


In [ ]:
import html
import json
import os
import re
import statistics
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from tqdm.auto import tqdm

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None

warnings.filterwarnings('ignore')
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
def ensure_directory(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

CURRENT_DIR = Path.cwd().resolve()
REMOTE_DATASET_URL = 'https://raw.githubusercontent.com/svakulenk0/wu-llms-ss26/main/dataset_clean.csv'
DEFAULT_RIS_SEED_URLS = [
    'https://www.ris.bka.gv.at/GeltendeFassung.wxe?Abfrage=Bundesnormen&Gesetzesnummer=10004569',
    'https://www.ris.bka.gv.at/GeltendeFassung.wxe?Abfrage=Bundesnormen&Gesetzesnummer=10004570',
    'https://www.ris.bka.gv.at/GeltendeFassung.wxe?Abfrage=Bundesnormen&Gesetzesnummer=10004873',
    'https://www.ris.bka.gv.at/GeltendeFassung.wxe?Abfrage=Bundesnormen&Gesetzesnummer=10003940',
]


def resolve_code_dir(current_dir: Path) -> Path:
    if current_dir.name == 'code':
        return current_dir
    candidate = current_dir / 'code'
    if candidate.exists():
        return candidate
    return current_dir


def resolve_submission_root(code_dir: Path) -> Path:
    if code_dir.name == 'code':
        return code_dir.parent
    if (code_dir / 'code').exists():
        return code_dir
    return code_dir.parent


def ensure_local_dataset(submission_root: Path, remote_url: str, filename: str = 'dataset_clean.csv') -> Path:
    local_path = submission_root / filename
    if local_path.exists():
        return local_path
    try:
        response = requests.get(remote_url, timeout=30)
        response.raise_for_status()
        local_path.write_text(response.text, encoding='utf-8')
        print(f'Downloaded benchmark dataset to {local_path}')
        return local_path
    except Exception as exc:
        raise FileNotFoundError(
            f'Could not find {filename} in the submission root and the download also failed. Original error: {exc}'
        ) from exc


def ensure_seed_file(code_dir: Path, default_urls: list[str]) -> Path:
    seed_path = code_dir / 'ris_seed_urls.txt'
    if seed_path.exists():
        return seed_path
    seed_path.write_text('\n'.join(default_urls) + '\n', encoding='utf-8')
    print(f'Created seed file at {seed_path}')
    return seed_path


NOTEBOOK_DIR = resolve_code_dir(CURRENT_DIR)
SUBMISSION_ROOT = resolve_submission_root(NOTEBOOK_DIR)
RESULT_DIR = ensure_directory(SUBMISSION_ROOT / 'results')
LOCAL_ASSET_DIR = NOTEBOOK_DIR / '_rag_assets'
MODEL_CACHE_DIR = LOCAL_ASSET_DIR / 'model_cache'
CORPUS_ROOT = LOCAL_ASSET_DIR / 'rag_corpus'
RAW_CORPUS_DIR = CORPUS_ROOT / 'raw'
PROCESSED_CORPUS_DIR = CORPUS_ROOT / 'processed'
DATASET_PATH = ensure_local_dataset(NOTEBOOK_DIR, REMOTE_DATASET_URL)
RIS_SEED_URL_FILE = ensure_seed_file(NOTEBOOK_DIR, DEFAULT_RIS_SEED_URLS)
MODEL_NAME = 'Qwen/Qwen3-0.6B'
EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
RESULT_PATH = RESULT_DIR / 'rag_qwen3_0_6b.csv'
DEBUG_RESULT_PATH = RESULT_DIR / 'rag_qwen3_0_6b_debug.csv'
FAILURE_LOG_PATH = RESULT_DIR / 'rag_qwen3_0_6b_failures.json'
CHUNK_CACHE_PATH = LOCAL_ASSET_DIR / 'rag_qwen3_corpus.json'
EMBEDDING_CACHE_PATH = LOCAL_ASSET_DIR / 'rag_qwen3_embeddings.npy'
RETRIEVAL_AUDIT_PATH = RESULT_DIR / 'rag_qwen3_0_6b_audit.jsonl'

AUTO_DOWNLOAD_RIS = True
REQUEST_TIMEOUT = 30
USER_AGENT = 'WU-LLM-Stage2-Notebook/1.0'
TOP_K = 2
CHUNK_SIZE = 420
CHUNK_OVERLAP = 64
DENSE_WEIGHT = 0.78
LEXICAL_WEIGHT = 0.22
DIRECT_FALLBACK_THRESHOLD = 0.72
BATCH_SIZE = 1
MAX_INPUT_TOKENS = 320
MAX_NEW_TOKENS = 48
TEMPERATURE = 0.1
TOP_P = 0.9
TOP_K_SAMPLING = 20
DO_SAMPLE = True
REPETITION_PENALTY = 1.10
ULTRAFAST_VALIDATION_MODE = True
ROW_LIMIT = None
FINAL_EXPORT = False
ALLOW_REMOTE_MISMATCH_FOR_SMOKE_TEST = True
MAX_RETRIES = 2
MIN_ANSWER_LENGTH = 18
WRITE_DEBUG_OUTPUT = True if ROW_LIMIT is not None else False
RESUME_FROM_EXISTING_RESULTS = False if ROW_LIMIT is not None else True
SEED = 42

set_seed(SEED)
ensure_directory(LOCAL_ASSET_DIR)
ensure_directory(MODEL_CACHE_DIR)
ensure_directory(RAW_CORPUS_DIR)
ensure_directory(PROCESSED_CORPUS_DIR)

print('Current directory:', CURRENT_DIR)
print('Notebook directory:', NOTEBOOK_DIR)
print('Submission root:', SUBMISSION_ROOT)
print('Dataset path:', DATASET_PATH)
print('RIS seed file:', RIS_SEED_URL_FILE)
print('Result path:', RESULT_PATH)
print('Generator model:', MODEL_NAME)
print('Embedding model:', EMBEDDING_MODEL_NAME)


In [ ]:
import csv
import html
import io
import json
import re
import statistics
import time
import warnings
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')


def ensure_directory(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def normalize_answer(text: str) -> str:
    text = text or ''
    return re.sub(r'\s+', ' ', text).strip()


def slugify(text: str) -> str:
    safe = re.sub(r'[^A-Za-z0-9._-]+', '-', text.strip())
    return safe.strip('-') or 'artifact'


def retry_call(fn, attempts: int = 2, wait_seconds: float = 1.5):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except Exception as exc:
            last_error = exc
            if attempt < attempts:
                time.sleep(wait_seconds * attempt)
    raise last_error


def read_benchmark_dataset(dataset_path: Path) -> list[dict]:
    if not dataset_path.exists():
        raise FileNotFoundError(f'Dataset not found: {dataset_path}')
    with dataset_path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        rows = []
        seen_ids = set()
        for row in reader:
            question_id = (row.get('id') or row.get('﻿id') or '').strip()
            prompt = (row.get('prompt') or '').strip()
            if not question_id or not prompt:
                continue
            if question_id in seen_ids:
                raise ValueError(f'Duplicate id detected: {question_id}')
            seen_ids.add(question_id)
            rows.append({'id': question_id, 'prompt': prompt})
    if not rows:
        raise ValueError('No usable rows were loaded from the dataset.')
    return rows


def fetch_remote_dataset_meta(remote_url: str) -> dict | None:
    try:
        response = requests.get(remote_url, timeout=30)
        response.raise_for_status()
        reader = csv.DictReader(io.StringIO(response.text))
        rows = list(reader)
        return {'row_count': len(rows), 'fieldnames': [name.lstrip('﻿') for name in (reader.fieldnames or [])]}
    except Exception as exc:
        print(f'Remote dataset check skipped: {exc}')
        return None


def validate_dataset(rows: list[dict], remote_meta: dict | None, final_export: bool, allow_smoke_test_mismatch: bool):
    if remote_meta is None:
        print('Remote dataset validation was skipped. Local validation still passed.')
        return
    local_count = len(rows)
    remote_count = remote_meta.get('row_count')
    if remote_count is None:
        return
    if local_count != remote_count:
        message = f'Local dataset has {local_count} rows but the course GitHub dataset reports {remote_count}.'
        if final_export or not allow_smoke_test_mismatch:
            raise ValueError(message)
        print(f'Warning: {message}')
    else:
        print(f'Row-count check passed: {local_count} rows.')


def load_existing_submission(result_path: Path) -> dict[str, str]:
    if not result_path.exists():
        return {}
    with result_path.open('r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        rows = {}
        for row in reader:
            question_id = (row.get('id') or row.get('﻿id') or '').strip()
            answer = normalize_answer(row.get('answer') or '')
            if question_id and answer:
                rows[question_id] = answer
        return rows


def write_submission(rows: list[tuple[str, str]], result_path: Path):
    ensure_directory(result_path.parent)
    with result_path.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['id', 'answer'])
        writer.writerows(rows)


def validate_submission(dataset_rows: list[dict], answers: dict[str, str], min_answer_length: int = 20):
    dataset_ids = [row['id'] for row in dataset_rows]
    missing_ids = [qid for qid in dataset_ids if qid not in answers]
    extra_ids = [qid for qid in answers if qid not in set(dataset_ids)]
    blank_ids = [qid for qid, answer in answers.items() if not normalize_answer(answer)]
    short_ids = [qid for qid, answer in answers.items() if len(normalize_answer(answer)) < min_answer_length]
    if missing_ids:
        raise ValueError(f'Submission is missing ids. First examples: {missing_ids[:5]}')
    if extra_ids:
        raise ValueError(f'Submission contains extra ids. First examples: {extra_ids[:5]}')
    if blank_ids:
        raise ValueError(f'Submission contains blank answers. First examples: {blank_ids[:5]}')
    return {'row_count': len(answers), 'short_answer_ids': short_ids}


In [ ]:
SYSTEM_PROMPT = (
    'Beantworte die oesterreichische Steuerfrage auf Deutsch in hoechstens zwei kurzen Saetzen. '
    'Nutze nur den gegebenen RIS-Kontext. '
    'Wenn eine Aussage klar passt, gib sie moeglichst eng am Kontext wieder. '
    'Kein Vorspann, keine Meta-Saetze, keine Wiederholung der Frage, keine erfundenen Fakten.'
)

STOPWORD_SET = {
    'die', 'der', 'das', 'und', 'oder', 'ist', 'sind', 'eine', 'einer', 'eines', 'einem',
    'einen', 'ein', 'im', 'in', 'den', 'dem', 'des', 'zu', 'auf', 'von', 'mit', 'fuer',
    'fur', 'nach', 'bei', 'als', 'nur', 'auch', 'wie', 'was', 'wann', 'welche', 'welcher',
    'welches', 'wird', 'werden', 'gilt', 'gelten', 'kann', 'koennen', 'konnen', 'nicht'
}


def pick_device():
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def describe_source(block: dict) -> str:
    source_name = block.get('source', '')
    if '10004569' in source_name:
        law_name = 'KStG'
    elif '10004570' in source_name:
        law_name = 'EStG'
    elif '10004873' in source_name:
        law_name = 'UStG'
    elif '10003940' in source_name:
        law_name = 'BAO'
    else:
        law_name = source_name
    if block.get('paragraph'):
        return f"{law_name}, {block['paragraph']}"
    return law_name


def build_prompt(question: str, context_blocks: list[dict]) -> str:
    cleaned_question = normalize_answer(question)
    if not context_blocks:
        return (
            'Kontext aus RIS:\nKein brauchbarer Treffer gefunden.\n\n'
            f'Frage:\n{cleaned_question}\n\n'
            'Gib eine kurze Antwort und sage, dass der geladene Kontext nicht ausreicht.'
        )
    context_block_list = []
    for item_index, block in enumerate(context_blocks, start=1):
        context_block_list.append(
            f"[{item_index}] Rechtsquelle: {describe_source(block)}\n"
            f"Normtext: {block['text']}"
        )
    context_text = '\n\n'.join(context_block_list)
    return (
        f'Kontext aus RIS:\n{context_text}\n\n'
        f'Frage:\n{cleaned_question}\n\n'
        'Gib eine kurze Antwort, die eng am Kontext bleibt.'
    )


def read_seed_urls(seed_file: Path) -> list[str]:
    if not seed_file.exists():
        return []
    urls = []
    for line in seed_file.read_text(encoding='utf-8').splitlines():
        cleaned = line.strip()
        if cleaned and not cleaned.startswith('#'):
            urls.append(cleaned)
    return urls


def download_ris_sources(seed_urls: list[str], raw_dir: Path):
    if not seed_urls:
        print('No RIS seed URLs found. The notebook will rely on local source files only.')
        return []
    downloaded = []
    headers = {'User-Agent': USER_AGENT}
    for index, url in enumerate(seed_urls, start=1):
        file_stem = slugify(Path(url).name or f'ris-{index}')
        output_path = raw_dir / f'{index:02d}_{file_stem}.html'
        if output_path.exists():
            downloaded.append(output_path)
            continue
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            output_path.write_text(response.text, encoding='utf-8')
            downloaded.append(output_path)
        except Exception as exc:
            print(f'RIS download failed for {url}: {exc}')
    return downloaded


def clean_ris_text(text: str) -> str:
    cleaned = html.unescape(text or '')
    cleaned = cleaned.replace('\xa0', ' ')
    cleaned = re.sub(r'\bAccesskey\b\s*\d+', ' ', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\b(?:Seitenbereiche|Navigationsleiste|Kontakt|Impressum|Datenschutzerkl?rung|Barrierefreiheitserkl?rung|Sitemap|English|Druckansicht|Andere Formate):?\b', ' ', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\b(?:StF|CELEX-Nr\.|Typ|Kurztitel|Index|Dokumentnummer)\b.*?(?=(?:\u00A7\s*\d+|Artikel\s+[IVXLC]+|$))', ' ', cleaned, flags=re.IGNORECASE | re.DOTALL)
    cleaned = re.sub(r'\b(?:Anmerkung|Beachte)\b.*?(?=(?:\u00A7\s*\d+|Artikel\s+[IVXLC]+|Absatz\s+eins|$))', ' ', cleaned, flags=re.IGNORECASE | re.DOTALL)
    cleaned = re.sub(r'\b(?:Paragraph\s+[A-Za-z??????]+,\s*)', ' ', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\b(?:Absatz|Ziffer|Litera|Strichaufz?hlung)\b\s+[a-z0-9]+', ' ', cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def extract_ris_norm_blocks(raw_html: str) -> list[str]:
    soup = BeautifulSoup(raw_html, 'html.parser')
    for node in soup.select('.sr-only, .onlyScreenreader, script, style, nav, header, footer'):
        node.decompose()

    blocks = []
    seen = set()
    for text_container in soup.select("div[id*='TextContainer']"):
        text_container_id = text_container.get('id', '') or ''
        if any(skip_token in text_container_id for skip_token in ['LangtitelContainer', 'AenderungContainer', 'PraeambelContainer']):
            continue
        raw_text = text_container.get_text('\n', strip=True)
        if not raw_text or raw_text.startswith('Langtitel') or 'Inhaltsverzeichnis' in raw_text:
            continue
        headings = ' '.join(node.get_text(' ', strip=True) for node in text_container.select('h4, h5'))
        combined = clean_ris_text(f"{headings}\n{raw_text}")
        if combined.lower().startswith('text '):
            combined = combined[5:].strip()
        if len(combined) < 120:
            continue
        if not re.search(r'\u00A7\s*\d+', combined):
            continue
        if combined not in seen:
            seen.add(combined)
            blocks.append(combined)
    return blocks


def html_to_text(raw_html: str) -> str:
    norm_blocks = extract_ris_norm_blocks(raw_html)
    if norm_blocks:
        return '\n\n'.join(norm_blocks)
    soup = BeautifulSoup(raw_html, 'html.parser')
    for node in soup.select('.sr-only, .onlyScreenreader, script, style, nav, header, footer'):
        node.decompose()
    return clean_ris_text(soup.get_text('\n'))


def read_source_file(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {'.txt', '.md'}:
        return path.read_text(encoding='utf-8', errors='ignore')
    if suffix in {'.html', '.htm'}:
        return html_to_text(path.read_text(encoding='utf-8', errors='ignore'))
    if suffix == '.pdf':
        if PdfReader is None:
            raise RuntimeError('PDF support is unavailable. Install pypdf first.')
        reader = PdfReader(str(path))
        return '\n\n'.join(normalize_answer(page.extract_text() or '') for page in reader.pages if normalize_answer(page.extract_text() or ''))
    return ''


def prepare_processed_corpus(raw_dir: Path, processed_dir: Path) -> list[Path]:
    processed_files = []
    source_files = sorted(path for path in raw_dir.rglob('*') if path.is_file())
    for source_path in source_files:
        try:
            text = read_source_file(source_path)
        except Exception as exc:
            print(f'Could not read {source_path.name}: {exc}')
            continue
        if not normalize_answer(text):
            continue
        target_path = processed_dir / f'{source_path.stem}.txt'
        target_path.write_text(text, encoding='utf-8')
        processed_files.append(target_path)
    return processed_files


def split_text_into_sections(text: str) -> list[str]:
    normalized = text or ''
    if not normalized.strip():
        return []
    raw_sections = re.split(r'\n\s*\n+', normalized)
    sections = [normalize_answer(section) for section in raw_sections if len(normalize_answer(section)) >= 80]
    return sections or [normalize_answer(normalized)]


def split_section_into_passages(section: str) -> list[str]:
    normalized = normalize_answer(section)
    if not normalized:
        return []

    paragraph_candidates = re.split(r'(?=(?:\u00A7\s*\d+[a-zA-Z]*|Artikel\s+[IVXLC]+))', normalized)
    paragraph_candidates = [normalize_answer(item) for item in paragraph_candidates if normalize_answer(item)]
    if not paragraph_candidates:
        paragraph_candidates = [normalized]

    passages = []
    for paragraph_text in paragraph_candidates:
        sentence_list = [normalize_answer(item) for item in re.split(r'(?<=[.!?;:])\s+', paragraph_text) if normalize_answer(item)]
        if not sentence_list:
            sentence_list = [paragraph_text]

        current_passage = ''
        for sentence in sentence_list:
            candidate = normalize_answer(f'{current_passage} {sentence}') if current_passage else sentence
            if current_passage and len(candidate) > CHUNK_SIZE:
                if len(current_passage) >= 60:
                    passages.append(current_passage)
                current_passage = sentence
            else:
                current_passage = candidate

        if current_passage and len(current_passage) >= 60:
            passages.append(current_passage)

    deduplicated = []
    seen = set()
    for passage in passages:
        key = passage.lower()
        if key not in seen:
            seen.add(key)
            deduplicated.append(passage)
    return deduplicated


def build_or_load_chunk_cache(processed_dir: Path, cache_path: Path) -> list[dict]:
    expected_config = {
        'chunk_size': CHUNK_SIZE,
        'chunk_overlap': CHUNK_OVERLAP,
        'source_count': len(list(processed_dir.glob('*.txt'))),
        'chunking_version': 2,
    }
    if cache_path.exists():
        payload = json.loads(cache_path.read_text(encoding='utf-8'))
        if isinstance(payload, dict) and payload.get('config') == expected_config and isinstance(payload.get('chunks'), list):
            return payload['chunks']

    chunks = []
    for text_file in sorted(processed_dir.glob('*.txt')):
        source_name = text_file.name
        section_list = split_text_into_sections(text_file.read_text(encoding='utf-8', errors='ignore'))
        for section_index, section in enumerate(section_list, start=1):
            passage_list = split_section_into_passages(section)
            for passage_index, piece in enumerate(passage_list, start=1):
                paragraph_match = re.search(r'§\s*\d+[a-zA-Z]*', piece)
                chunks.append({
                    'source': source_name,
                    'section': section_index,
                    'passage': passage_index,
                    'paragraph': paragraph_match.group(0) if paragraph_match else None,
                    'text': piece,
                    'search_text': f"{describe_source({'source': source_name, 'paragraph': paragraph_match.group(0) if paragraph_match else None})} {piece}".strip(),
                })

    if not chunks:
        raise ValueError('No usable RAG chunks were created. Add RIS sources before running generation.')

    payload = {'config': expected_config, 'chunks': chunks}
    cache_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    return chunks


def normalize_word_list(text: str) -> list[str]:
    token_list = re.findall(r'[A-Za-z0-9]+', str(text).lower())
    return [token for token in token_list if len(token) >= 4 and token not in STOPWORD_SET]


def build_embedding_cache(rag_chunks: list[dict], embedding_cache_path: Path):
    embedding_device = 'cpu'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        embedding_device = 'mps'
    elif torch.cuda.is_available():
        embedding_device = 'cuda'

    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=embedding_device)
    search_text_list = [chunk['search_text'] for chunk in rag_chunks]

    if embedding_cache_path.exists():
        matrix = np.load(embedding_cache_path)
        if len(matrix) == len(search_text_list):
            return embedder, matrix

    matrix = embedder.encode(search_text_list, normalize_embeddings=True, show_progress_bar=True)
    matrix = np.asarray(matrix, dtype=np.float32)
    np.save(embedding_cache_path, matrix)
    return embedder, matrix


def retrieve_passages(question_text: str, rag_chunks: list[dict], embedding_matrix, token_sets: list[set], embedder, top_k: int) -> list[dict]:
    query_embedding = embedder.encode([question_text], normalize_embeddings=True, show_progress_bar=False)[0]
    query_embedding = np.asarray(query_embedding, dtype=np.float32)
    dense_score_array = embedding_matrix @ query_embedding
    query_token_set = set(normalize_word_list(question_text))

    lexical_score_list = []
    for token_set in token_sets:
        if not query_token_set:
            lexical_score_list.append(0.0)
            continue
        lexical_score_list.append(len(query_token_set & token_set) / max(1, len(query_token_set)))

    lexical_score_array = np.asarray(lexical_score_list, dtype=np.float32)
    hybrid_score_array = (DENSE_WEIGHT * dense_score_array) + (LEXICAL_WEIGHT * lexical_score_array)
    best_index_list = np.argsort(hybrid_score_array)[-max(12, top_k * 3):][::-1]

    result_list = []
    seen_key_set = set()
    for idx in best_index_list:
        row = dict(rag_chunks[int(idx)])
        key = (row['source'], row.get('paragraph'), row['text'])
        if key in seen_key_set:
            continue
        seen_key_set.add(key)
        row['dense_score'] = float(dense_score_array[int(idx)])
        row['lexical_score'] = float(lexical_score_array[int(idx)])
        row['score'] = float(hybrid_score_array[int(idx)])
        result_list.append(row)
        if len(result_list) >= top_k:
            break
    return result_list


def render_model_prompt(tokenizer, prompt: str) -> str:
    cleaned_prompt = normalize_answer(prompt)
    if getattr(tokenizer, 'chat_template', None):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': cleaned_prompt},
        ]
        if MODEL_NAME.startswith('Qwen/Qwen3'):
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f'Instruction: {SYSTEM_PROMPT}\nQuestion: {cleaned_prompt}\nAnswer:'


def load_model_and_tokenizer(model_name: str):
    try:
        device = pick_device()
        if device.type == 'cpu':
            torch.set_num_threads(max(1, (os.cpu_count() or 2) - 1))

        tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=str(MODEL_CACHE_DIR), use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model_kwargs = {'cache_dir': str(MODEL_CACHE_DIR), 'low_cpu_mem_usage': True}
        if device.type in {'cuda', 'mps'}:
            model_kwargs['torch_dtype'] = 'auto'
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        model.to(device)
        model.eval()
        return model, tokenizer, device
    except Exception as exc:
        raise RuntimeError(f'Could not load model {model_name}. Check the model name, internet connection, and local cache. Original error: {exc}') from exc


def clean_generated_answer(text: str) -> str:
    compact_text = normalize_answer(str(text or '').replace('<think>', ' ').replace('</think>', ' '))
    if not compact_text:
        return compact_text
    compact_text = re.sub(r'\[(?:Quelle|Source)\s*\d+:[^\]]*\]', ' ', compact_text, flags=re.IGNORECASE)
    compact_text = re.sub(r'\b\S*Gesetzesnummer-\d+\S*\.txt\b', ' ', compact_text, flags=re.IGNORECASE)
    compact_text = re.sub(r'^(?:\u00A7\s*\d+[a-zA-Z]*\s*)?(?:Abs\.?\s*\d+\s*)?(?:Z(?:iffer)?\s*\d+\s*)?(?:lit\.?\s*[a-z]\s*)+', '', compact_text, flags=re.IGNORECASE)
    compact_text = re.sub(r'^(?:\d+\s*)?(?:Z(?:iffer)?\s*\d+\s*)?(?:lit\.?\s*[a-z]\s*)+', '', compact_text, flags=re.IGNORECASE)
    compact_text = re.sub(r'^[-,:;.)\]]+\s*', '', compact_text)
    compact_text = normalize_answer(compact_text)

    sentence_list = []
    for piece in compact_text.replace('!', '.').replace('?', '.').split('.'):
        piece = normalize_answer(piece)
        if piece:
            sentence_list.append(piece)

    unique_sentence_list = []
    seen_set = set()
    for sentence in sentence_list:
        key = sentence.lower()
        if key not in seen_set:
            seen_set.add(key)
            unique_sentence_list.append(sentence)

    cleaned_text = '. '.join(unique_sentence_list[:2]).strip()
    if cleaned_text and cleaned_text[-1] not in '.!?':
        cleaned_text += '.'
    return cleaned_text


def build_retrieval_fallback(retrieved_row_list: list[dict]) -> str:
    if not retrieved_row_list:
        return 'Der geladene Kontext reicht fuer eine belastbare Antwort nicht aus.'
    fallback_text = clean_generated_answer(retrieved_row_list[0]['text'])
    if looks_fragmentary(fallback_text):
        return 'Der geladene Kontext reicht fuer eine belastbare Antwort nicht aus.'
    return fallback_text or 'Der geladene Kontext reicht fuer eine belastbare Antwort nicht aus.'


def looks_fragmentary(text: str) -> bool:
    cleaned = clean_generated_answer(text)
    lowered = cleaned.lower()
    if not cleaned:
        return True
    if len(cleaned) < 30:
        return True
    if re.match(r'^(?:\d+|ziffer|lit\.?|abs\.?|satz|und\b|oder\b|sowie\b)', lowered):
        return True
    if cleaned[0].islower():
        return True
    return False


def build_rewrite_prompt(question_text: str, retrieved_row_list: list[dict]) -> str:
    if not retrieved_row_list:
        return (
            'Kontext aus RIS:\nKein brauchbarer Treffer gefunden.\n\n'
            f'Frage:\n{normalize_answer(question_text)}\n\n'
            'Antworte kurz und sage, dass der Kontext nicht ausreicht.'
        )
    top_blocks = retrieved_row_list[:2]
    context_text = '\n\n'.join(
        f"[{index}] Rechtsquelle: {describe_source(block)}\nNormtext: {block['text']}"
        for index, block in enumerate(top_blocks, start=1)
    )
    return (
        f'Kontext aus RIS:\n{context_text}\n\n'
        f'Frage:\n{normalize_answer(question_text)}\n\n'
        'Formuliere eine kurze, vollstaendige Antwort in ein bis zwei ganzen Saetzen. '
        'Keine Ziffernfolgen, keine Satzfragmente, keine Listenreste, kein wortwoertliches Kopieren von Paragraphenanfaengen.'
    )


def generate_rewritten_fallback(model, tokenizer, device, question_text: str, retrieved_row_list: list[dict]) -> str:
    rewrite_prompt = build_rewrite_prompt(question_text, retrieved_row_list)
    model_prompt = render_model_prompt(tokenizer, rewrite_prompt)
    model_inputs = tokenizer([model_prompt], return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
    model_inputs = {key: value.to(device) for key, value in model_inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max(40, min(64, MAX_NEW_TOKENS)),
            temperature=0.05,
            top_p=0.9,
            top_k=10,
            do_sample=True,
            repetition_penalty=max(REPETITION_PENALTY, 1.12),
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_length = int(model_inputs['attention_mask'].sum(dim=1).tolist()[0])
    answer_ids = generated_ids[0][prompt_length:]
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
    return clean_generated_answer(answer_text)


def looks_weak(answer_text: str, retrieved_row_list: list[dict]) -> bool:
    text = clean_generated_answer(answer_text)
    lowered = text.lower()
    if not text:
        return True
    if looks_fragmentary(text):
        return True
    if 'kontext' in lowered or 'ris' in lowered or 'frage' in lowered or 'quelle' in lowered:
        return True
    if '?' in text:
        return True
    answer_token_set = set(normalize_word_list(text))
    retrieved_token_set = set()
    for row in retrieved_row_list:
        retrieved_token_set |= set(normalize_word_list(row['text']))
    if answer_token_set and len(answer_token_set & retrieved_token_set) <= 1:
        return True
    return False


def generate_answer(model, tokenizer, device, question_text: str, retrieved_row_list: list[dict]) -> str:
    prompt = build_prompt(question_text, retrieved_row_list)
    model_prompt = render_model_prompt(tokenizer, prompt)
    model_inputs = tokenizer([model_prompt], return_tensors='pt', padding=True, truncation=True, max_length=MAX_INPUT_TOKENS)
    model_inputs = {key: value.to(device) for key, value in model_inputs.items()}
    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K_SAMPLING,
            do_sample=DO_SAMPLE,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    prompt_length = int(model_inputs['attention_mask'].sum(dim=1).tolist()[0])
    answer_ids = generated_ids[0][prompt_length:]
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True)
    return clean_generated_answer(answer_text)


def choose_final_answer(model, tokenizer, device, question_text: str, retrieved_row_list: list[dict]):
    if not retrieved_row_list:
        return 'Der geladene Kontext reicht fuer eine belastbare Antwort nicht aus.', 'no_context'

    best_retrieved_score = float(retrieved_row_list[0]['score'])
    generated_answer = generate_answer(model, tokenizer, device, question_text, retrieved_row_list)
    if not looks_weak(generated_answer, retrieved_row_list):
        return generated_answer, 'generator'

    rewritten_fallback = generate_rewritten_fallback(model, tokenizer, device, question_text, retrieved_row_list)
    if not looks_weak(rewritten_fallback, retrieved_row_list):
        return rewritten_fallback, 'retrieval_rewrite'

    if best_retrieved_score >= DIRECT_FALLBACK_THRESHOLD:
        fallback_text = build_retrieval_fallback(retrieved_row_list)
        if fallback_text != 'Der geladene Kontext reicht fuer eine belastbare Antwort nicht aus.':
            return fallback_text, 'direct_retrieval'

    return 'Der geladene Kontext reicht fuer eine belastbare Antwort nicht aus.', 'retrieval_fallback'


In [ ]:
full_benchmark_rows = read_benchmark_dataset(DATASET_PATH)
remote_meta = fetch_remote_dataset_meta(REMOTE_DATASET_URL)
validate_dataset(full_benchmark_rows, remote_meta, FINAL_EXPORT, ALLOW_REMOTE_MISMATCH_FOR_SMOKE_TEST)

benchmark_rows = full_benchmark_rows[:ROW_LIMIT] if ROW_LIMIT is not None else full_benchmark_rows

seed_urls = read_seed_urls(RIS_SEED_URL_FILE)
if AUTO_DOWNLOAD_RIS:
    download_ris_sources(seed_urls, RAW_CORPUS_DIR)

processed_files = prepare_processed_corpus(RAW_CORPUS_DIR, PROCESSED_CORPUS_DIR)
if not processed_files:
    raise ValueError('No processed RAG corpus files were created. Add local source files to rag_corpus/raw or enable RIS downloading.')

rag_chunks = build_or_load_chunk_cache(PROCESSED_CORPUS_DIR, CHUNK_CACHE_PATH)
embedder, corpus_embedding_matrix = build_embedding_cache(rag_chunks, EMBEDDING_CACHE_PATH)
corpus_token_set_list = [set(normalize_word_list(chunk['search_text'])) for chunk in rag_chunks]

print('Benchmark rows:', len(benchmark_rows))
print('Processed corpus files:', len(processed_files))
print('Chunk count:', len(rag_chunks))
print('Embedding matrix shape:', tuple(corpus_embedding_matrix.shape))
pd.DataFrame(rag_chunks)[['source', 'paragraph', 'text']].head(3)


In [ ]:
model, tokenizer, device = load_model_and_tokenizer(MODEL_NAME)
existing_answers = load_existing_submission(RESULT_PATH) if RESUME_FROM_EXISTING_RESULTS else {}
pending_rows = [row for row in benchmark_rows if row['id'] not in existing_answers]

retrieval_audit = []
debug_rows = []
timings = []
failures = {}

for row in tqdm(pending_rows, desc='Hybrid RAG generation'):
    question_text = normalize_answer(row['prompt'])
    started = time.perf_counter()
    try:
        retrieved_row_list = retry_call(
            lambda: retrieve_passages(
                question_text,
                rag_chunks,
                corpus_embedding_matrix,
                corpus_token_set_list,
                embedder,
                TOP_K,
            ),
            attempts=MAX_RETRIES,
        )
        answer_text, answer_mode = retry_call(
            lambda: choose_final_answer(model, tokenizer, device, question_text, retrieved_row_list),
            attempts=MAX_RETRIES,
        )
        cleaned_answer = normalize_answer(answer_text)
        if not cleaned_answer:
            raise ValueError('The merged RAG pipeline returned an empty answer after cleaning.')
        existing_answers[row['id']] = cleaned_answer
    except Exception as exc:
        failures[row['id']] = str(exc)
        retrieved_row_list = []
        cleaned_answer = ''
        answer_mode = 'failed'
    timings.append(time.perf_counter() - started)

    retrieval_audit.append({
        'id': row['id'],
        'question': question_text,
        'mode': answer_mode,
        'retrieved_sources': [describe_source(item) for item in retrieved_row_list],
        'retrieved_scores': [round(float(item.get('score', 0.0)), 4) for item in retrieved_row_list],
        'retrieved_dense_scores': [round(float(item.get('dense_score', 0.0)), 4) for item in retrieved_row_list],
        'retrieved_lexical_scores': [round(float(item.get('lexical_score', 0.0)), 4) for item in retrieved_row_list],
        'retrieved_snippets': [item['text'][:220] for item in retrieved_row_list],
    })

    if WRITE_DEBUG_OUTPUT:
        debug_rows.append({
            'id': row['id'],
            'answer': cleaned_answer,
            'mode': answer_mode,
            'retrieved_sources': ' | '.join(describe_source(item) for item in retrieved_row_list),
            'retrieved_scores': ' | '.join(f"{float(item.get('score', 0.0)):.4f}" for item in retrieved_row_list),
            'retrieved_snippets': ' | '.join(item['text'][:180] for item in retrieved_row_list),
        })

    ordered_rows = [(dataset_row['id'], existing_answers[dataset_row['id']]) for dataset_row in benchmark_rows if dataset_row['id'] in existing_answers]
    write_submission(ordered_rows, RESULT_PATH)
    FAILURE_LOG_PATH.write_text(json.dumps(failures, indent=2, ensure_ascii=False), encoding='utf-8')
    with RETRIEVAL_AUDIT_PATH.open('w', encoding='utf-8') as handle:
        for item in retrieval_audit:
            handle.write(json.dumps(item, ensure_ascii=False) + '\n')
    if WRITE_DEBUG_OUTPUT:
        pd.DataFrame(debug_rows, columns=['id', 'answer', 'mode', 'retrieved_sources', 'retrieved_scores', 'retrieved_snippets']).to_csv(
            DEBUG_RESULT_PATH,
            index=False,
            encoding='utf-8-sig',
        )

print('Hybrid RAG generation loop finished.')


In [ ]:
validation = validate_submission(benchmark_rows, existing_answers, min_answer_length=MIN_ANSWER_LENGTH)
summary = {
    'rows_written': validation['row_count'],
    'failed_rows': len(failures),
    'average_row_seconds': round(statistics.mean(timings), 3) if timings else 0.0,
    'short_answer_count': len(validation['short_answer_ids']),
    'result_path': str(RESULT_PATH),
    'debug_result_path': str(DEBUG_RESULT_PATH) if WRITE_DEBUG_OUTPUT else None,
    'retrieval_audit_path': str(RETRIEVAL_AUDIT_PATH),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('\nShort answers worth reviewing:', validation['short_answer_ids'][:10])
pd.read_csv(RESULT_PATH, encoding='utf-8-sig').head(5)


In [ ]:
if WRITE_DEBUG_OUTPUT and DEBUG_RESULT_PATH.exists():
    pd.read_csv(DEBUG_RESULT_PATH, encoding='utf-8-sig').head(5)
elif RETRIEVAL_AUDIT_PATH.exists():
    pd.read_json(RETRIEVAL_AUDIT_PATH, lines=True)[['id', 'mode', 'retrieved_sources', 'retrieved_scores']].head(5)
else:
    print('No debug or audit file exists yet.')


## Final note

Dieses Notebook schreibt die Haupt-CSV, die optionale Debug-CSV und das Audit im Modellordner. Technische Hilfsdaten landen im lokalen Unterordner `_rag_assets`.
